<a href="https://colab.research.google.com/github/MithunSrinivas28/wafer-defect-ai/blob/final/Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

BLOCK 1 — IMPORTS

In [89]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [90]:
import tensorflow as tf
import numpy as np
import os
import shutil
import csv
from datetime import datetime


In [91]:
# ===== BASE PATH =====

BASE_DIR = "/content/drive/MyDrive/Wafer_Pipeline"

MODEL_PATH = BASE_DIR + "/wafer_xai2_model.keras"


In [92]:
# ===== CREATE MAIN FOLDERS =====

folders = [
    BASE_DIR,
    BASE_DIR + "/Input_Images",
    BASE_DIR + "/Results",
    BASE_DIR + "/Datasets/Self-learning",
    BASE_DIR + "/Offline_Storage/Pending_Sync"
]

for f in folders:
    os.makedirs(f, exist_ok=True)

print("Folder structure created")


Folder structure created


In [93]:
# ===== LOAD MODEL =====

model = tf.keras.models.load_model(MODEL_PATH)

print("Model loaded successfully")


Model loaded successfully


In [94]:
# ===== CLASS NAMES (MUST MATCH TRAINING ORDER) =====

CLASS_NAMES = [
    'bridge','clean','cmp','crack',
    'ler','open','others','vias'
]

print("Classes:", CLASS_NAMES)


Classes: ['bridge', 'clean', 'cmp', 'crack', 'ler', 'open', 'others', 'vias']


#  IMAGE PREPROCESSING

In [95]:
# ===== IMAGE PREPROCESSING =====

IMG_SIZE = 224


def preprocess_image(img_path):

    img = tf.keras.utils.load_img(
        img_path,
        color_mode="grayscale",
        target_size=(IMG_SIZE, IMG_SIZE)
    )

    img = tf.keras.utils.img_to_array(img)

    # Normalize
    img = img / 255.0

    # Gray → RGB
    img = tf.repeat(img, 3, axis=-1)

    # Add batch dimension
    img = tf.expand_dims(img, axis=0)

    return img


In [96]:
# List files in Input_Images folder

input_files = os.listdir(BASE_DIR + "/Input_Images")

print("Images found:", input_files)


Images found: ['crack.png']


In [97]:
# ===== SINGLE IMAGE PREDICTION =====

def predict_image(img_path):

    img = preprocess_image(img_path)

    preds = model.predict(img, verbose=0)[0]

    class_id = np.argmax(preds)

    confidence = float(np.max(preds))

    label = CLASS_NAMES[class_id]

    return label, confidence


In [111]:
# ===== TEST ON ONE IMAGE =====

test_image = BASE_DIR + "/Input_Images/bri.png"

label, conf = predict_image(test_image)

print("Prediction :", label)
print("Confidence :", round(conf*100, 2), "%")


Prediction : bridge
Confidence : 53.73 %
